# 

In [1]:
import pandas as pd
import numpy as np
import torch

import utils
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
complete_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike.csv')

# Remove LCLid MAC001654
complete_df = complete_df[complete_df['LCLid'] != 'MAC001654']

print(complete_df.shape)
complete_df.head()

(81920326, 17)


,LCLid,tstp,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid,spike_type,spike_magnitude,temperature,windSpeed,humidity
0,MAC000002,2013-01-01 00:00:00,0.226083,0.219,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0,0.33250,0.3672,0.6494
1,MAC000002,2013-01-01 00:30:00,0.222500,0.241,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0,0.33885,0.3689,0.6429
2,MAC000002,2013-01-01 01:00:00,0.220000,0.191,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0,0.34520,0.3706,0.6364
3,MAC000002,2013-01-01 01:30:00,0.208250,0.235,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0,0.34085,0.3784,0.6104
4,MAC000002,2013-01-01 02:00:00,0.198167,0.182,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0,0.33650,0.3862,0.5844


In [4]:
# # A dataframe for the [mean, median, std, min, max, gradient] for all the users
# statistics_df = complete_df.groupby('LCLid').agg({
#     'mean': 'first',
#     'median': 'first',
#     'std': 'first',
#     'min': 'first',
#     'max': 'first',
#     'gradient': 'first'
# })
# statistics_df = statistics_df.reset_index()
# statistics_df.head()
# statistics_df.to_csv('../../Data_processed/Statistics.csv', index=False)
# del statistics_df

In [5]:
complete_df = utils.cyclic_time(complete_df)
complete_df.head()

,LCLid,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid,spike_type,spike_magnitude,temperature,windSpeed,humidity,date_sin,date_cos,time_sin,time_cos
0,MAC000002,0.2261,0.219,0.2525,0.158,0.2471,0.0,2.994,0.1093,10,0,0,1.0,0.3325,0.3672,0.6494,0.0172,0.9999,0.0000,1.0000
1,MAC000002,0.2225,0.241,0.2525,0.158,0.2471,0.0,2.994,0.1093,10,0,0,1.0,0.3388,0.3689,0.6429,0.0172,0.9999,0.1305,0.9914
2,MAC000002,0.2200,0.191,0.2525,0.158,0.2471,0.0,2.994,0.1093,10,0,0,1.0,0.3452,0.3706,0.6364,0.0172,0.9999,0.2588,0.9659
3,MAC000002,0.2082,0.235,0.2525,0.158,0.2471,0.0,2.994,0.1093,10,0,0,1.0,0.3408,0.3784,0.6104,0.0172,0.9999,0.3827,0.9239
4,MAC000002,0.1982,0.182,0.2525,0.158,0.2471,0.0,2.994,0.1093,10,0,0,1.0,0.3365,0.3862,0.5844,0.0172,0.9999,0.5000,0.8660


In [6]:
kmedoid_df = complete_df[complete_df['is_medoid'] == 1]
regular_df = complete_df[complete_df['is_medoid'] == 0]

kmedoid_df = kmedoid_df.drop(['is_medoid'], axis=1)
regular_df = regular_df.drop(['is_medoid'], axis=1)

# Use 200 users for training, 100 for testing
ids = regular_df['LCLid'].unique()
np.random.seed(42)
train_ids = np.random.choice(ids, 2000 , replace=False)
test_ids = np.setdiff1d(ids, train_ids)
test_ids = np.random.choice(test_ids, 100, replace=False)

medoid_to_spike_df = regular_df[regular_df['LCLid'].isin(train_ids)]
test_df = regular_df[regular_df['LCLid'].isin(test_ids)]

print(kmedoid_df.shape)
print(medoid_to_spike_df.shape)
print(test_df.shape)

(344618, 19)
(33916616, 19)
(1682533, 19)


In [7]:
kmedoid_df.to_csv('../../Data_processed/Kmedoids_with_Weather_Espike_medoids.csv', index=False)
medoid_to_spike_df.to_csv('../../Data_processed/Kmedoids_with_Weather_Espike_medoid_to_spike.csv', index=False)
test_df.to_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv', index=False)

del complete_df
del test_df

In [8]:
sequence_length = 12

# Extract relevant columns
weather_columns = ['temperature', 'windSpeed', 'humidity']
cluster_columns = ['kmedoid_clusters']
time_columns = ['date_sin', 'date_cos', 'time_sin', 'time_cos']
statistical_columns = ['mean', 'median', 'std',  'min', 'max', 'gradient']
spike_columns = ['spike_type', 'spike_magnitude']
energy_column = ['energy(kWh/hh)']

def create_sequences(df, sequence_length):
    weather_sequences = []
    cluster_sequences = []
    time_sequences = []
    statistical_sequences = []
    spike_sequences = []
    energy_sequences = []

    LCLids = df['LCLid'].unique()

    for LCLid in tqdm(LCLids, desc=f"Processing LCLids"):  # Wrap LCLids with tqdm for a progress bar
        df_group = df[df['LCLid'] == LCLid]

        weather_data = df_group[weather_columns].values
        cluster_data = df_group[cluster_columns].values
        time_data = df_group[time_columns].values
        statistical_data = df_group[statistical_columns].values
        spike_data = df_group[spike_columns].values
        energy_data = df_group[energy_column].values

        i = 0
        while i < len(weather_data) - sequence_length:
            weather = weather_data[i:i + sequence_length]
            cluster = cluster_data[i:i + sequence_length]
            time = time_data[i:i + sequence_length]
            statistical = statistical_data[i:i + sequence_length]
            spike = spike_data[i:i + sequence_length]
            energy = energy_data[i:i + sequence_length]

            weather_sequences.append(weather)
            cluster_sequences.append(cluster)
            time_sequences.append(time)
            statistical_sequences.append(statistical)
            spike_sequences.append(spike)
            energy_sequences.append(energy)

            n = np.random.randint(1, sequence_length)
            i += n

    return np.array(weather_sequences), np.array(cluster_sequences), np.array(time_sequences), np.array(statistical_sequences), np.array(spike_sequences), np.array(energy_sequences)

weather_data, cluster_data, time_data, statistical_data, spike_data, energy_data = create_sequences(kmedoid_df, sequence_length)
torch.save(weather_data, '../../Data_processed/VAE/weather_data.pt')
torch.save(cluster_data, '../../Data_processed/VAE/cluster_data.pt')
torch.save(time_data, '../../Data_processed/VAE/time_data.pt')
torch.save(statistical_data, '../../Data_processed/VAE/statistical_data.pt')
torch.save(spike_data, '../../Data_processed/VAE/spike_data.pt')
torch.save(energy_data, '../../Data_processed/VAE/energy_data.pt')
# Free up memory
del weather_data, cluster_data, time_data, statistical_data, spike_data, energy_data
del kmedoid_df

weather_data_m2s, cluster_data_m2s, time_data_m2s, statistical_data_m2s, spike_data_m2s, energy_data_m2s = create_sequences(medoid_to_spike_df, sequence_length)
torch.save(weather_data_m2s, '../../Data_processed/VAE/weather_data_m2s.pt', pickle_protocol=4)
torch.save(cluster_data_m2s, '../../Data_processed/VAE/cluster_data_m2s.pt', pickle_protocol=4)
torch.save(time_data_m2s, '../../Data_processed/VAE/time_data_m2s.pt', pickle_protocol=4) 
torch.save(statistical_data_m2s, '../../Data_processed/VAE/statistical_data_m2s.pt', pickle_protocol=4)
torch.save(spike_data_m2s, '../../Data_processed/VAE/spike_data_m2s.pt', pickle_protocol=4)
torch.save(energy_data_m2s, '../../Data_processed/VAE/energy_data_m2s.pt', pickle_protocol=4)
# Free up memory
del weather_data_m2s, cluster_data_m2s, time_data_m2s, statistical_data_m2s, spike_data_m2s, energy_data_m2s
del medoid_to_spike_df

Processing LCLids: 100%|██████████| 2000/2000 [34:19<00:00,  1.03s/it]
